# AFL Player Data Cleaning, Validation & Merge




## 1. Load raw data

In [4]:
import pandas as pd
import numpy as np
import json

info_raw = pd.read_csv("../../afl_players_info_raw.csv")
stats_raw = pd.read_csv("../../afl_players_seasonal_stats_raw.csv", low_memory=False)

print("players_info_raw:", info_raw.shape)
print("seasonal_stats_raw:", stats_raw.shape)


players_info_raw: (2848, 16)
seasonal_stats_raw: (25491, 54)


In [5]:
info_raw.head()

,id,player_name,player_full_name,first_name,last_name,born_date,debut_date,debut_age,last_date,last_age,height,weight,profile_pic,player_link,player_common_names,player_teams
0,43261,Ryan Abbott,Ryan_Abbott,Ryan,Abbott,1991-06-25,2018-08-02,27,2020-09-05,29,200,100,NaN,https://afltables.com/afl/stats/players/R/Ryan...,NaN,"{Geelong Cats,St Kilda Saints}"
1,43262,Gary Ablett,Gary_Ablett1,Gary,Ablett,1984-05-14,2002-03-30,17,2020-10-24,36,182,87,NaN,https://afltables.com/afl/stats/players/G/Gary...,NaN,"{Geelong Cats,Gold Coast Suns}"
2,43276,Leek Aleer,Leek_Aleer,Leek,Aleer,2001-08-21,2022-07-30,20,2025-09-06,24,195,85,https://res.cloudinary.com/dijzdikkh/image/upl...,https://afltables.com/afl/stats/players/L/Leek...,NaN,NaN
3,43264,Nathan Ablett,Nathan_Ablett,Nathan,Ablett,1985-12-13,2005-08-12,19,2011-08-28,25,195,95,NaN,https://afltables.com/afl/stats/players/N/Nath...,NaN,"{Geelong Cats,Gold Coast Suns}"
4,43265,Cain Ackland,Cain_Ackland,Cain,Ackland,1982-03-16,2001-03-30,19,2008-05-16,26,196,101,NaN,https://afltables.com/afl/stats/players/C/Cain...,NaN,"{Carlton Blues,Port Adelaide Power,St Kilda Sa..."


In [6]:
stats_raw.head()

,player_id,year,team,is_finals,games_played,kicks,marks,handballs,disposals,goals,...,avg_contested_possessions,avg_uncontested_possessions,avg_contested_marks,avg_marks_inside_50,avg_one_percenters,avg_bounces,avg_goal_assists,avg_score,avg_fantasy_points,avg_percentage_played
0,43261,2018,Geelong Cats,False,3,17.0,9.0,21.0,38.0,3.0,...,6.7,6.7,0.3,1.7,4.3,0.0,0.3,6.7,92.7,84.3
1,43261,2018,Geelong Cats,True,1,3.0,2.0,2.0,5.0,0.0,...,2.0,3.0,1.0,1.0,5.0,0.0,0.0,1.0,61.0,81.0
2,43261,2019,Geelong Cats,False,1,5.0,3.0,6.0,11.0,1.0,...,6.0,8.0,0.0,1.0,3.0,0.0,0.0,6.0,74.0,83.0
3,43261,2020,St Kilda Saints,False,1,2.0,2.0,1.0,3.0,1.0,...,0.0,3.0,0.0,1.0,0.0,0.0,0.0,6.0,22.0,54.0
4,43262,2002,Geelong Cats,False,12,37.0,13.0,63.0,100.0,10.0,...,5.7,3.0,0.3,0.3,1.0,0.2,0.0,5.3,36.9,NaN


In [7]:
info_raw.dtypes

id                      int64
player_name            object
player_full_name       object
first_name             object
last_name              object
born_date              object
debut_date             object
debut_age               int64
last_date              object
last_age                int64
height                  int64
weight                  int64
profile_pic            object
player_link            object
player_common_names    object
player_teams           object
dtype: object

In [8]:
stats_raw.dtypes

player_id                       object
year                             int64
team                            object
is_finals                         bool
games_played                     int64
kicks                          float64
marks                          float64
handballs                      float64
disposals                      float64
goals                          float64
behinds                        float64
hit_outs                       float64
tackles                        float64
rebound_50s                    float64
inside_50s                     float64
clearances                     float64
clangers                       float64
free_kicks_for                 float64
free_kicks_against             float64
brownlow_votes                 float64
contested_possessions          float64
uncontested_possessions        float64
contested_marks                float64
marks_inside_50                float64
one_percenters                 float64
bounces                  

## 2. Data Quality Assessment (raw data)

 Now I will profile both datasets to identify concrete issues: duplicates, missing
values, malformed keys, impossible values (e.g. negative counts, 0 kg weight), and inconsistent
categorical text (team names).

In [9]:
dqa = {}

dqa['info_full_duplicate_rows'] = int(info_raw.duplicated().sum())
dqa['info_duplicate_ids'] = int(info_raw['id'].duplicated().sum())
dqa['stats_full_duplicate_rows'] = int(stats_raw.duplicated().sum())
dqa['stats_malformed_player_id'] = int(stats_raw['player_id'].astype(str).str.startswith('ID_').sum())
dqa['info_weight_zero'] = int((info_raw['weight'] == 0).sum())
dqa['stats_negative_games_played'] = int((stats_raw['games_played'] < 0).sum())
dqa['stats_team_name_variants'] = int(stats_raw['team'].nunique())
dqa['stats_team_name_variants_normalized'] = int(stats_raw['team'].str.strip().str.title().nunique())
dqa['info_null_profile_pic'] = int(info_raw['profile_pic'].isna().sum())
dqa['info_null_player_common_names'] = int(info_raw['player_common_names'].isna().sum())
dqa['info_null_player_teams'] = int(info_raw['player_teams'].isna().sum())

print("Data Quality Assessment (raw):")
for k, v in dqa.items():
    print(f"  {k}: {v}")


Data Quality Assessment (raw):
  info_full_duplicate_rows: 5
  info_duplicate_ids: 5
  stats_full_duplicate_rows: 10
  stats_malformed_player_id: 10
  info_weight_zero: 2
  stats_negative_games_played: 8
  stats_team_name_variants: 114
  stats_team_name_variants_normalized: 20
  info_null_profile_pic: 2211
  info_null_player_common_names: 2773
  info_null_player_teams: 94


## 3. Cleaning log

Every fix is recorded programmatically as we go, so the rationale is auditable.

In [10]:
cleaning_log = []

def log(table, issue, action, rationale, n_affected):
    cleaning_log.append({
        "table": table,
        "issue": issue,
        "action": action,
        "rationale": rationale,
        "rows_affected": n_affected,
    })


## 4. Clean `players_info`

In [11]:
# Fix 1: Drop exact duplicate rows in players_info
info = info_raw.copy()
rows_before_info = len(info)

n = int(info.duplicated().sum())
info = info.drop_duplicates()
log("players_info", "Exact duplicate rows (same id + all fields)",
    "Dropped duplicate rows, kept first occurrence",
    "True duplicates add no information and would double-count players in joins",
    n)
print(f"Dropped {n} duplicate rows from players_info")


Dropped 5 duplicate rows from players_info


In [12]:
# Fix 2: confirm id is now a clean unique key
dup_ids_remaining = info[info.duplicated(subset='id', keep=False)]
log("players_info", "Duplicate player id values after exact-dup removal",
    f"Checked: {len(dup_ids_remaining)} rows remain with shared id (0 expected)",
    "Confirms 'id' is a clean unique key for the player dimension table after de-duplication",
    len(dup_ids_remaining))
print("Remaining duplicate ids:", len(dup_ids_remaining))

def normalize_player_teams(s):
    if pd.isna(s):
        return s
    teams = [t.strip().title() for t in str(s).strip("{}").split(",") if t.strip()]
    return "{" + ",".join(teams) + "}"

info['player_teams'] = info['player_teams'].apply(normalize_player_teams)


Remaining duplicate ids: 0


In [13]:
# Fix 3: weight == 0 kg is physically impossible -> treat as missing
n = int((info['weight'] == 0).sum())
info.loc[info['weight'] == 0, 'weight'] = np.nan
log("players_info", "weight = 0 kg",
    "Replaced 0 with NaN (missing)",
    "0 kg is physically impossible for an AFL player; treated as a missing/placeholder value rather than dropping the player record",
    n)
print(f"Fixed {n} weight=0 rows")


Fixed 2 weight=0 rows


In [14]:
# Fix 4: standardise text fields (trim whitespace, convert literal 'NULL' strings to NaN)
text_cols = ['player_name', 'player_full_name', 'first_name', 'last_name', 'player_teams']
for c in text_cols:
    info[c] = info[c].astype(str).str.strip().replace({'nan': np.nan, 'NULL': np.nan})

log("players_info", "Leading/trailing whitespace and literal 'NULL' strings in text fields",
    f"Trimmed whitespace and converted literal 'NULL' strings to proper NaN in {text_cols}",
    "Literal 'NULL' strings are not recognised as missing by pandas by default and break groupby/merge keys",
    rows_before_info)
print("Standardised text columns:", text_cols)


Standardised text columns: ['player_name', 'player_full_name', 'first_name', 'last_name', 'player_teams']


In [15]:
# Fix 5: parse date columns
for c in ['born_date', 'debut_date', 'last_date']:
    info[c] = pd.to_datetime(info[c], errors='coerce')

n_bad_dates = int(info[['born_date', 'debut_date', 'last_date']].isna().sum().sum())
log("players_info", "Date columns stored as text",
    "Parsed born_date, debut_date, last_date to proper datetime dtype",
    "Enables date arithmetic/validation and consistent downstream typing",
    n_bad_dates)
print("Unparseable dates after conversion:", n_bad_dates)


Unparseable dates after conversion: 0


In [16]:
# Fix 6: sanity-check debut_age against born_date/debut_date
calc_debut_age = ((info['debut_date'] - info['born_date']).dt.days / 365.25).round().astype('Int64')
mismatch_debut = int((calc_debut_age != info['debut_age']).sum())
log("players_info", "debut_age consistency check vs born_date/debut_date",
    f"Compared stored debut_age to age calculated from dates; {mismatch_debut} rows differ by rounding only (kept stored values)",
    "Differences were all within 1 year (rounding), so original recorded ages were retained instead of overwritten",
    mismatch_debut)
print("debut_age rounding mismatches (informational only):", mismatch_debut)


debut_age rounding mismatches (informational only): 1392


In [17]:
# Fix 7: document remaining legitimate missingness (not imputed)
n_teams = int(info['player_teams'].isna().sum())
log("players_info", "Missing player_teams",
    "Left as NaN (genuinely unknown - no club history recorded)",
    "Cannot be reliably inferred from other columns; imputing a team would fabricate data",
    n_teams)

n_pics = int(info['profile_pic'].isna().sum())
log("players_info", "Missing profile_pic / player_common_names",
    "Left as NaN",
    "Cosmetic/optional metadata fields not required for statistical analysis; imputing is not meaningful",
    n_pics)

rows_after_info = len(info)
print(f"players_info rows: {rows_before_info} -> {rows_after_info}")


players_info rows: 2848 -> 2843


In [18]:
info.head()

,id,player_name,player_full_name,first_name,last_name,born_date,debut_date,debut_age,last_date,last_age,height,weight,profile_pic,player_link,player_common_names,player_teams
0,43261,Ryan Abbott,Ryan_Abbott,Ryan,Abbott,1991-06-25,2018-08-02,27,2020-09-05,29,200,100.0,NaN,https://afltables.com/afl/stats/players/R/Ryan...,NaN,"{Geelong Cats,St Kilda Saints}"
1,43262,Gary Ablett,Gary_Ablett1,Gary,Ablett,1984-05-14,2002-03-30,17,2020-10-24,36,182,87.0,NaN,https://afltables.com/afl/stats/players/G/Gary...,NaN,"{Geelong Cats,Gold Coast Suns}"
2,43276,Leek Aleer,Leek_Aleer,Leek,Aleer,2001-08-21,2022-07-30,20,2025-09-06,24,195,85.0,https://res.cloudinary.com/dijzdikkh/image/upl...,https://afltables.com/afl/stats/players/L/Leek...,NaN,NaN
3,43264,Nathan Ablett,Nathan_Ablett,Nathan,Ablett,1985-12-13,2005-08-12,19,2011-08-28,25,195,95.0,NaN,https://afltables.com/afl/stats/players/N/Nath...,NaN,"{Geelong Cats,Gold Coast Suns}"
4,43265,Cain Ackland,Cain_Ackland,Cain,Ackland,1982-03-16,2001-03-30,19,2008-05-16,26,196,101.0,NaN,https://afltables.com/afl/stats/players/C/Cain...,NaN,"{Carlton Blues,Port Adelaide Power,St Kilda Sa..."


## 5. Clean `seasonal_stats`

In [19]:
stats = stats_raw.copy()
rows_before_stats = len(stats)

# Fix 1: exact duplicate rows and drop them 
n = int(stats.duplicated().sum())
stats = stats.drop_duplicates()
log("seasonal_stats", "Exact duplicate rows",
    "Dropped duplicate rows, kept first occurrence",
    "Duplicate season records would double-count games/stats for a player-year-team-finals combination",
    n)
print(f"Dropped {n} duplicate rows from seasonal_stats")


Dropped 10 duplicate rows from seasonal_stats


In [20]:
# Fix 2: malformed player_id values with 'ID_' text prefix 
mask_bad_id = stats['player_id'].astype(str).str.startswith('ID_')
n = int(mask_bad_id.sum())
stats['player_id'] = stats['player_id'].astype(str).str.replace('ID_', '', regex=False)
log("seasonal_stats", "player_id stored with malformed 'ID_' text prefix",
    "Stripped 'ID_' prefix so player_id is purely numeric",
    "Prefix is an encoding artifact; underlying numeric value matches valid ids in players_info, confirmed by successful join after the fix",
    n)
print(f"Fixed {n} malformed player_id values")


Fixed 10 malformed player_id values


In [21]:
# Fix 3: cast player_id to a consistent numeric dtype for merging 
stats['player_id'] = pd.to_numeric(stats['player_id'], errors='coerce').astype('Int64')
n_unparseable = int(stats['player_id'].isna().sum())
log("seasonal_stats", "player_id dtype",
    "Converted player_id to nullable integer (Int64) to match players_info.id",
    "player_id was loaded as text/object due to mixed formatting; consistent dtype is required for a reliable merge",
    n_unparseable)
print("Unparseable player_id after cast:", n_unparseable)


Unparseable player_id after cast: 0


In [22]:
# Fix 4: negative games_played is impossible -> drop corrupted rows
n = int((stats['games_played'] < 0).sum())
display_cols = ['player_id', 'year', 'team', 'games_played']
print("Rows with negative games_played:")
print(stats.loc[stats['games_played'] < 0, display_cols])

stats = stats[stats['games_played'] >= 0].copy()
log("seasonal_stats", "Negative games_played values",
    "Dropped rows with negative games_played",
    "games_played cannot be negative; these rows are corrupted source data with no reliable way to recover the true value, and they were also duplicated rows, so removal avoids skewing season totals/averages",
    n)
print(f"Dropped {n} rows with negative games_played")


Rows with negative games_played:
       player_id  year               team  games_played
19642      43260  2020   Richmond Tigers            -13
19643      43260  2020   Richmond Tigers             -1
25030      43260  2021   Richmond Tigers            -21
25031      43260  2022   Richmond Tigers             -7
Dropped 4 rows with negative games_played


In [23]:
# Fix 5: standardise team name formatting
n_before_variants = stats['team'].nunique()
stats['team'] = stats['team'].astype(str).str.strip().str.title()
stats['team'] = stats['team'].str.replace(r'\s+', ' ', regex=True).str.strip()
n_after_variants = stats['team'].nunique()

log("seasonal_stats", "Inconsistent team name formatting (extra whitespace, ALL CAPS, lowercase)",
    f"Standardised to trimmed Title Case, reducing {n_before_variants} raw variants to {n_after_variants} distinct team names",
    "Same club appeared under multiple text variants (e.g. 'WEST COAST EAGLES', ' West Coast Eagles ', 'West Coast Eagles'); standardising prevents the same team being treated as different groups in aggregations",
    rows_before_stats)
print(f"Team name variants: {n_before_variants} -> {n_after_variants}")
sorted(stats['team'].unique())


Team name variants: 114 -> 20


['Adelaide Crows',
 'Brisbane Bears',
 'Brisbane Lions',
 'Carlton Blues',
 'Collingwood Magpies',
 'Essendon Bombers',
 'Fitzroy Lions',
 'Fremantle Dockers',
 'Geelong Cats',
 'Gold Coast Suns',
 'Greater Western Sydney Giants',
 'Hawthorn Hawks',
 'Melbourne Demons',
 'North Melbourne Kangaroos',
 'Port Adelaide Power',
 'Richmond Tigers',
 'St Kilda Saints',
 'Sydney Swans',
 'West Coast Eagles',
 'Western Bulldogs']

In [24]:
# Fix 6: document missingness in advanced stat columns (structural, not imputed)
stat_null_cols = [c for c in stats.columns if stats[c].isna().any()]
n_total_nulls = int(stats[stat_null_cols].isna().sum().sum())

log("seasonal_stats", "Missing values in advanced statistical columns (e.g. contested_possessions, hit_outs, bounces, brownlow_votes)",
    "Left as NaN rather than imputing with 0 or mean",
    "Missingness is structural: these metrics were not officially tracked/recorded in earlier AFL seasons (e.g. bounces ~100% missing pre-1992) or are sparse by nature (brownlow_votes). Filling with 0 would falsely imply 'recorded zero' instead of 'not tracked'; an indicator is preserved by leaving NaN",
    n_total_nulls)

print("Columns with missing values and their share of missing rows:")
(stats[stat_null_cols].isna().mean().sort_values(ascending=False) * 100).round(1)


Columns with missing values and their share of missing rows:


avg_goal_assists               27.8
goal_assists                   27.8
brownlow_votes                 27.5
total_percentage_played        24.5
avg_percentage_played          24.5
avg_hit_outs                   22.4
hit_outs                       22.4
avg_bounces                    22.2
bounces                        22.2
avg_marks_inside_50            20.3
marks_inside_50                20.3
contested_marks                19.3
avg_contested_marks            19.3
one_percenters                 14.3
avg_one_percenters             14.3
rebound_50s                    13.9
avg_rebound_50s                13.9
avg_clearances                 13.5
clearances                     13.5
contested_possessions          13.1
avg_contested_possessions      13.1
avg_uncontested_possessions    13.0
uncontested_possessions        13.0
avg_inside_50s                 11.7
inside_50s                     11.7
avg_clangers                   11.5
clangers                       11.5
avg_behinds                 

In [25]:
# Confirm the missingness is concentrated in earlier seasons (era effect, not random)
stats.groupby('year')['bounces'].apply(lambda x: x.isna().mean()).round(2)


year
1983    1.00
1984    1.00
1985    1.00
1986    1.00
1987    1.00
1988    1.00
1989    1.00
1990    1.00
1991    1.00
1992    0.99
1993    0.98
1994    0.99
1995    0.99
1996    0.99
1997    0.99
1998    0.99
1999    0.39
2000    0.32
2001    0.27
2002    0.21
2003    0.19
2004    0.17
2005    0.13
2006    0.10
2007    0.08
2008    0.08
2009    0.05
2010    0.05
2011    0.03
2012    0.02
2013    0.01
2014    0.01
2015    0.01
2016    0.01
2017    0.01
2018    0.01
2019    0.01
2020    0.01
2021    0.01
2022    0.01
2023    0.01
2024    0.45
2025    0.36
Name: bounces, dtype: float64

In [26]:
# Fix 7: negative total_fantasy_points - validated as plausible, not an error
n_neg_fantasy = int((stats['total_fantasy_points'] < 0).sum())
print(stats.loc[stats['total_fantasy_points'] < 0,
                 ['player_id', 'year', 'total_fantasy_points', 'games_played']])

log("seasonal_stats", "Negative total_fantasy_points",
    "Retained as-is (no change)",
    "AFL Fantasy scoring can legitimately produce negative totals from a single low-impact game (heavy clangers/free-kicks-against, few possessions); values were sense-checked against games_played=1 and are plausible, not data errors",
    n_neg_fantasy)


       player_id  year  total_fantasy_points  games_played
344        45896  1998                    -3             1
3302       43812  2013                    -6             1
7215       44401  2022                    -3             1
10017      44846  2021                    -2             1
11860      45147  2019                    -1             1
15754      45632  1999                    -3             1
19511      46095  2013                    -6             1
21704      45744  2003                    -4             1
23614      45756  2007                    -1             1
23773      45945  2001                    -2             1


In [27]:
rows_after_stats = len(stats)
print(f"seasonal_stats rows: {rows_before_stats} -> {rows_after_stats}")
stats.head()


seasonal_stats rows: 25491 -> 25477


,player_id,year,team,is_finals,games_played,kicks,marks,handballs,disposals,goals,...,avg_contested_possessions,avg_uncontested_possessions,avg_contested_marks,avg_marks_inside_50,avg_one_percenters,avg_bounces,avg_goal_assists,avg_score,avg_fantasy_points,avg_percentage_played
0,43261,2018,Geelong Cats,False,3,17.0,9.0,21.0,38.0,3.0,...,6.7,6.7,0.3,1.7,4.3,0.0,0.3,6.7,92.7,84.3
1,43261,2018,Geelong Cats,True,1,3.0,2.0,2.0,5.0,0.0,...,2.0,3.0,1.0,1.0,5.0,0.0,0.0,1.0,61.0,81.0
2,43261,2019,Geelong Cats,False,1,5.0,3.0,6.0,11.0,1.0,...,6.0,8.0,0.0,1.0,3.0,0.0,0.0,6.0,74.0,83.0
3,43261,2020,St Kilda Saints,False,1,2.0,2.0,1.0,3.0,1.0,...,0.0,3.0,0.0,1.0,0.0,0.0,0.0,6.0,22.0,54.0
4,43262,2002,Geelong Cats,False,12,37.0,13.0,63.0,100.0,10.0,...,5.7,3.0,0.3,0.3,1.0,0.2,0.0,5.3,36.9,NaN


## 6. Merge datasets

Left join `seasonal_stats` (the fact table) onto `players_info` (the dimension table) on
`player_id` / `id`, so every season record is retained even if biographical info is
unavailable for that player.

In [28]:
merged = stats.merge(
    info,
    left_on='player_id',
    right_on='id',
    how='left',
    suffixes=('', '_info'),
    indicator=True
)

unmatched = merged[merged['_merge'] == 'left_only']
n_unmatched = int(len(unmatched))
unmatched_ids = sorted(unmatched['player_id'].dropna().unique().tolist())
n_unmatched_unique_ids = len(unmatched_ids)

log("merged", "seasonal_stats rows with no matching players_info record",
    f"Kept as left join (rows retained with NaN info columns); {n_unmatched} rows affected",
    "Preserves all season-level statistical records for analysis even when biographical info is unavailable, rather than silently discarding game data; unmatched ids are reported separately for follow-up/data sourcing",
    n_unmatched)

merged = merged.drop(columns=['id', '_merge'])

print(f"Merged shape: {merged.shape}")
print(f"Unmatched rows: {n_unmatched} ({n_unmatched_unique_ids} distinct player_ids)")
unmatched_ids[:20]


Merged shape: (25477, 69)
Unmatched rows: 400 (266 distinct player_ids)


[43280,
 43292,
 43299,
 43319,
 43325,
 43329,
 43332,
 43348,
 43359,
 43361,
 43363,
 43364,
 43365,
 43378,
 43379,
 43396,
 43410,
 43420,
 43426,
 43433]

## 8. Save cleaned outputs

In [29]:
info.to_csv("players_info_cleaned.csv", index=False)
stats.to_csv("seasonal_stats_cleaned.csv", index=False)
merged.to_csv("merged_players.csv", index=False)

cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv("cleaning_log.csv", index=False)

print("Saved: players_info_cleaned.csv, seasonal_stats_cleaned.csv, merged_players.csv,")
print("       cleaning_log.csv, validation_report.json, data_quality_assessment.json")


Saved: players_info_cleaned.csv, seasonal_stats_cleaned.csv, merged_players.csv,
       cleaning_log.csv, validation_report.json, data_quality_assessment.json


In [30]:
cleaning_log_df

,table,issue,action,rationale,rows_affected
0,players_info,Exact duplicate rows (same id + all fields),"Dropped duplicate rows, kept first occurrence",True duplicates add no information and would d...,5
1,players_info,Duplicate player id values after exact-dup rem...,Checked: 0 rows remain with shared id (0 expec...,Confirms 'id' is a clean unique key for the pl...,0
2,players_info,weight = 0 kg,Replaced 0 with NaN (missing),0 kg is physically impossible for an AFL playe...,2
3,players_info,Leading/trailing whitespace and literal 'NULL'...,Trimmed whitespace and converted literal 'NULL...,Literal 'NULL' strings are not recognised as m...,2848
4,players_info,Date columns stored as text,"Parsed born_date, debut_date, last_date to pro...",Enables date arithmetic/validation and consist...,0
5,players_info,debut_age consistency check vs born_date/debut...,Compared stored debut_age to age calculated fr...,"Differences were all within 1 year (rounding),...",1392
6,players_info,Missing player_teams,Left as NaN (genuinely unknown - no club histo...,Cannot be reliably inferred from other columns...,93
7,players_info,Missing profile_pic / player_common_names,Left as NaN,Cosmetic/optional metadata fields not required...,2207
8,seasonal_stats,Exact duplicate rows,"Dropped duplicate rows, kept first occurrence",Duplicate season records would double-count ga...,10
9,seasonal_stats,player_id stored with malformed 'ID_' text prefix,Stripped 'ID_' prefix so player_id is purely n...,Prefix is an encoding artifact; underlying num...,10


## 9. Observations & insights from the cleaned dataset

In [31]:
print("Seasons covered:", merged['year'].min(), "-", merged['year'].max())
print("Distinct clubs:", merged['team'].nunique())
print("Distinct players in merged dataset:", merged['player_id'].nunique())

career_games = merged.groupby('player_id')['games_played'].sum().sort_values(ascending=False)
print("\nTop 5 players by total games played in dataset:")
print(career_games.head(5))


Seasons covered: 1983 - 2025
Distinct clubs: 20
Distinct players in merged dataset: 3108

Top 5 players by total games played in dataset:
player_id
45576    432
44673    425
43488    407
45522    400
43406    387
Name: games_played, dtype: int64


In [32]:
merged['decade'] = (merged['year'] // 10) * 10
merged.groupby('decade')['avg_goals'].mean().round(2)


decade
1980    1.80
1990    1.58
2000    0.99
2010    0.55
2020    0.67
Name: avg_goals, dtype: float64

### Key observations

1. **Coverage** : the cleaned dataset spans 1983–2025 across 20 AFL clubs and ~3,100 distinct
   players, giving a long enough time series for trend analysis.
2. **Goals-per-game has declined over time** - average goals per game fell from ~1.8 in the
   1980s to under 0.7 in the 2010s/2020s, consistent with the league's well-documented shift
   toward congested, possession-based football and away from high-scoring full-forwards.
3. **Advanced metrics are era-limited** - stats like `bounces`, `contested_possessions`, and
   `hit_outs` are only reliably available from the early-1990s/2000s onward. Any model or
   comparison using these fields should either filter to recent seasons or explicitly handle
   the missingness rather than imputing zeros.
4. **266 players (400 season rows, ~1.6% of records) have statistics but no biographical
   profile** - these are likely players whose `players_info` record was never scraped/loaded.
   They were kept in the merged dataset (with blank bio fields) rather than dropped, since the
   performance data itself is still valid and useful; the unmatched ids are listed in
   `validation_report.json` for follow-up data sourcing.
5. **Source data had encoding/scraping artifacts**, not just normal missingness - a literal
   `"ID_"` text prefix on some `player_id` values and inconsistent club-name casing both point
   to upstream data being assembled from multiple scraped sources without standardisation.
   These were fixed at the format level rather than treated as content errors.
